<span id="reference-tracking과-optimal-control"></span>

# Reference Tracking and Optimal Control

<span id="학습-목표"></span>

## Learning objectives

Examine why reference error remains even with a stable controller, and compare tracking performance with control-input cost.

<span id="목표값을-따라가도록-reference-이득-정하기"></span>

## Choosing reference gain for reference tracking

<span id="시작-조건"></span>

### Starting conditions

$A=\begin{bmatrix}0&1\\-2&-3\end{bmatrix}$, $B=\begin{bmatrix}0\\1\end{bmatrix}$, $C=\begin{bmatrix}1&0\end{bmatrix}$, $K=\begin{bmatrix}6&3\end{bmatrix}$, $r=1$, x(0)=0.

<span id="진행-방법"></span>

### Procedure

1.  Use $N=1$ and $N=8$ in turn to calculate unit-step responses on the same time axis. Record the responses and steady-state outputs side by side.
2.  For $0=(A-BK)x_{ss}+BNr$ under $r=1$, derive $x_{ss}$ in terms of $N$. Use $y_{ss}=Cx_{ss}$ to find $N$ satisfying $y_{ss}=1$, and show your calculations.
3.  Compare the directly calculated $N$ with the final values of both responses, and explain what the reference gain changed.

<span id="남길-결과"></span>

### Results to retain

- Both step responses and steady-state values, and the equation used to obtain N.

In [ ]:
import numpy as np
from scipy.signal import place_poles,StateSpace,lsim
from scipy.linalg import solve_continuous_are
import matplotlib.pyplot as plt
A=np.array([[0.,1.],[-2.,-3.]])
B=np.array([[0.],[1.]]);C=np.array([[1.,0.]])
K=place_poles(A,B,[-2.,-4.]).gain_matrix
N=float((-1/(C@np.linalg.solve(A-B@K,B))).item())
print('K:',K,'N:',N);assert np.isclose(N,8)
t=np.linspace(0,5,1001)
for gain in [1.,N]:
    _,y,_=lsim(StateSpace(A-B@K,B*gain,C,0),np.ones(len(t)),t)
    plt.plot(t,y,label=f'N={gain:g}')
plt.xlabel('Time [s]');plt.ylabel('Position');plt.legend();plt.grid()



<span id="결과-기록자가-점검"></span>

## Record results and self-check

**N:**

**Steady-state value:**

**Closed-loop Pole:**

<span id="확인할-점"></span>

### Points to check

- Do the directly calculated steady-state outputs match the final values of each simulation?
- Do closed-loop poles remain unchanged when N changes?

<span id="출력-오차와-입력-비용의-절충"></span>

## Trade-off between output error and input cost

<span id="시작-조건-1"></span>

### Starting conditions

$Q=\operatorname{diag}(q,1)$, $q\in\{1,25\}$, and $R=\begin{bmatrix}1\end{bmatrix}$. Use $x(0)=\begin{bmatrix}1\\0\end{bmatrix}$, $r=0$.

<span id="진행-방법-1"></span>

### Procedure

1.  Solve the CARE for each Q and calculate $K=R^{-1}B^\top P$.
2.  Plot position and control input on the same time axis.
3.  Compare the integral of squared position error and the integral of squared input separately.

<span id="남길-결과-1"></span>

### Results to retain

- Position and input plots and a comparison table of both integrals.

In [ ]:
from scipy.linalg import solve_continuous_are
for q in [1.,25.]:
    Q=np.diag([q,1.]);P=solve_continuous_are(A,B,Q,[[1.]])
    Kq=B.T@P
    _,x,_=lsim(StateSpace(A-B@Kq,np.zeros((2,1)),np.eye(2),np.zeros((2,1))),np.zeros(len(t)),t,X0=[1,0])
    u=-(x@Kq.T).ravel()
    print(q,Kq,np.trapezoid(x[:,0]**2,t),np.trapezoid(u**2,t))
    plt.plot(t,x[:,0],label=f'q={q:g}')
plt.xlabel('Time [s]');plt.ylabel('Position');plt.legend();plt.grid()
# TODO: Add control-input plots and pole checks.


<span id="결과-기록자가-점검-1"></span>

## Record results and self-check

**q:**

**K:**

**Position-error integral:**

**Input cost:**

<span id="확인할-점-1"></span>

### Points to check

- Are both Q matrices positive definite, with $R>0$?
- Do all poles of $A-BK$ have negative real parts?

<span id="모델-오차가-있을-때의-목표-추종"></span>

## Reference tracking with model error

<span id="시작-조건-2"></span>

### Starting conditions

Design using nominal B, and change only the actual input gain to $B_{\mathrm{actual}}=gB$ and $g\in\{.8,1,1.2\}$. Use $K=\begin{bmatrix}6&3\end{bmatrix}$, $N=8$, $r=1$.

<span id="진행-방법-2"></span>

### Procedure

1.  Find the poles and final step-response value for each actual model.
2.  Check whether N calculated from the nominal model also achieves the exact reference in every actual model.
3.  Distinguish the roles of reference-gain adjustment and integral-error feedback.

<span id="남길-결과-2"></span>

### Results to retain

- Three response curves and a table of final values and errors.

In [ ]:
K=np.array([[6.,3.]])
for g in [.8,1.,1.2]:
    Ba=g*B;Acl=A-Ba@K
    steady=float((-C@np.linalg.solve(Acl,Ba*8)).item())
    _,y,_=lsim(StateSpace(Acl,Ba*8,C,0),np.ones(len(t)),t)
    print(g,np.linalg.eigvals(Acl),steady);plt.plot(t,y,label=str(g))
plt.xlabel('Time [s]');plt.ylabel('Position');plt.legend();plt.grid()
# TODO: Explain why error remains and what integral feedback would additionally address.


<span id="결과-기록자가-점검-2"></span>

## Record results and self-check

**Input gain:**

**Pole:**

**Final value:**

**Tracking error:**

<span id="확인할-점-2"></span>

### Points to check

- Were both the feedback and reference terms multiplied by the actual B_actual?
- Did you check stability before calculating final values?

<span id="공식-예제데이터"></span>

## Official examples and data

- [SciPy · Pole placement](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.place_poles.html)
- [SciPy · Continuous-time CARE](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_continuous_are.html)